# Phase 3 — Evaluate the Model
AI Foundations Bootcamp | MNIST Capstone

> The test set is used **only once** in this notebook — final evaluation only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import os

np.random.seed(42)
os.makedirs('../reports', exist_ok=True)
print('Libraries loaded.')

## 1. Load Model & Test Data

In [ ]:
# Load saved model
model = keras.models.load_model('../models/mnist_cnn.h5')

# Load and preprocess test set (same steps as training)
(_, _), (X_test_raw, y_test) = keras.datasets.mnist.load_data()
X_test = X_test_raw / 255.0
X_test = X_test[..., np.newaxis]

print(f'Test set shape: {X_test.shape}')
print('Model loaded successfully.')

## 2. Test Accuracy

In [ ]:
# Evaluate on test set — ONE TIME ONLY
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')
print(f'Test Loss     : {test_loss:.4f}')

target = 0.99
status = '✓ Target MET' if test_acc >= target else '✗ Target not met'
print(f'\nTarget (≥99%) : {status}')

## 3. Classification Report (Precision, Recall, F1 per digit)

In [ ]:
y_pred = np.argmax(model.predict(X_test), axis=1)

report = classification_report(y_test, y_pred,
                                target_names=[str(i) for i in range(10)])
print('=== Classification Report ===')
print(report)

## 4. Confusion Matrix (10×10)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
disp = ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=list(range(10)),
    cmap='Blues',
    ax=ax
)
ax.set_title('Confusion Matrix — MNIST Test Set', fontsize=13)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.savefig('../reports/confusion_matrix.png', dpi=120)
plt.show()
print('Saved: reports/confusion_matrix.png')

## 5. Error Analysis — 9 Misclassified Images

In [ ]:
# Find all misclassified indices
wrong_idx = np.where(y_pred != y_test)[0]
print(f'Total errors: {len(wrong_idx)} out of {len(y_test)} ({len(wrong_idx)/len(y_test)*100:.2f}%)')

# Show first 9 errors in a grid
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for ax, idx in zip(axes.flat, wrong_idx[:9]):
    ax.imshow(X_test_raw[idx], cmap='gray')
    ax.set_title(f'True: {y_test[idx]}  |  Pred: {y_pred[idx]}',
                 color='red', fontsize=10)
    ax.axis('off')
plt.suptitle('Error Analysis — Misclassified Digits', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/error_analysis.png', dpi=120)
plt.show()
print('Saved: reports/error_analysis.png')

## 6. Error Analysis Commentary

Based on the confusion matrix and the sample misclassified images above:

- **4 vs 9**: The most frequent confusion. Both digits share a closed loop at the top; poorly written 4s with a closed top are read as 9.
- **3 vs 5**: The curves of 3 and 5 overlap when handwriting is hasty or asymmetric.
- **7 vs 1**: Slanted 1s or short-capped 7s are ambiguous.

These patterns confirm the model fails on *ambiguous* examples, not on clearly written digits — a reasonable limitation given the clean MNIST dataset.

## 7. Summary Dashboard (BONUS +2)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig)

# --- Panel 1: Confusion matrix ---
ax1 = fig.add_subplot(gs[0, 0])
im = ax1.imshow(cm, cmap='Blues')
ax1.set_title('Confusion Matrix')
ax1.set_xlabel('Predicted'); ax1.set_ylabel('True')
ax1.set_xticks(range(10)); ax1.set_yticks(range(10))
plt.colorbar(im, ax=ax1)

# --- Panel 2: Per-class accuracy ---
ax2 = fig.add_subplot(gs[0, 1])
per_class_acc = cm.diagonal() / cm.sum(axis=1)
bars = ax2.bar(range(10), per_class_acc, color='steelblue')
ax2.set_title('Per-Class Accuracy')
ax2.set_xlabel('Digit'); ax2.set_ylabel('Accuracy')
ax2.set_xticks(range(10))
ax2.set_ylim(0.97, 1.0)
ax2.axhline(0.99, color='red', linestyle='--', label='99% target')
ax2.legend()
for bar, val in zip(bars, per_class_acc):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.001,
             f'{val:.3f}', ha='center', va='top', fontsize=8, color='white')

# --- Panel 3: Error count by digit ---
ax3 = fig.add_subplot(gs[1, 0])
errors_per_class = cm.sum(axis=1) - cm.diagonal()
ax3.bar(range(10), errors_per_class, color='tomato')
ax3.set_title('Error Count per Digit')
ax3.set_xlabel('True Digit'); ax3.set_ylabel('Number of Errors')
ax3.set_xticks(range(10))

# --- Panel 4: Sample misclassified ---
ax4 = fig.add_subplot(gs[1, 1])
ax4.axis('off')
summary = (
    f'Test Accuracy : {test_acc*100:.2f}%\n'
    f'Total Errors  : {len(wrong_idx)}\n'
    f'Error Rate    : {len(wrong_idx)/len(y_test)*100:.2f}%\n\n'
    f'Hardest digit : {per_class_acc.argmin()} '
    f'({per_class_acc.min()*100:.1f}%)\n'
    f'Easiest digit : {per_class_acc.argmax()} '
    f'({per_class_acc.max()*100:.1f}%)'
)
ax4.text(0.05, 0.95, summary, transform=ax4.transAxes,
         fontsize=12, verticalalignment='top', family='monospace',
         bbox=dict(boxstyle='round', facecolor='#f0f4f8', alpha=0.8))
ax4.set_title('Model Summary')

plt.suptitle('MNIST CNN — Evaluation Dashboard', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/dashboard.png', dpi=120)
plt.show()
print('Saved: reports/dashboard.png')